# EMA 6611 – Optoelectronics Materials Processing (Fall 2026)
## Homework 2: Absorbers/Emitters and Carrier-Selective Junctions — Writing the Report as You Go

**Assigned:** Monday, September 28, 2026

**Due:** Sunday, October 11, 2026, 11:59 PM ET (submit on Webcourses)

**Total: 100 points** (+5 bonus)

---

**Name:** *(double-click to edit)*

**UCF ID:** *(double-click to edit)*

> **Before you start:** you opened this notebook from the course GitHub repository, which is read-only. Go to **File → Save a copy in Drive** *right now* so you are working in your own copy — otherwise your work will not be saved.


## Overview and Learning Objectives

Homework 1 gave you a topic, three datasheets, a 50-article corpus with a top-20, and a live Overleaf project. This assignment turns that material into the first three written pieces of your research report: the **Commercially Available Devices** section and the **Absorbers or Emitters** and **Carrier-Selective Junctions** subsections of the Discussion and Analysis. From here on, every homework has two halves that you do together: analysis in this notebook, and the corresponding report section written in Overleaf while the analysis is fresh. The notebook gives you tools (crystal-structure viewer, optical-constant loader, band-diagram solver) and structured tables; the report is where you interpret what they show for *your* device.

After completing this assignment, you will be able to:

1. Read a manufacturer datasheet critically and turn it into a cited table, an annotated figure and a comparison that identifies what the manufacturer does and does not disclose.
2. Describe the crystal structure of your absorber or emitter (lattice, basis, space group, lattice constants, alloy sublattice) and produce unit-cell and lattice-plane visualizations from a reproducible script.
3. Obtain the complex refractive index of your material from a public database, convert it to an absorption coefficient, and explain the absorption edge in terms of the bandgap value and type — or, for an emitter, connect the emission wavelength to composition.
4. Extract processing, microstructure, doping and defect information from your top-20 articles into a structured table and write a synthesis (not a paper-by-paper summary) from it.
5. Classify the electron- and hole-selective contacts of your device by the mechanism that makes them selective, and construct its equilibrium band diagram by the standard step-by-step procedure.

**Connection to your research report:** the three report sections you draft here are worth 40 of the report's 100 points (Table 1 of the *Research Report: Assignment and Rubric*). Table 2 of that document lists what each Discussion subsection must cover; the tasks below are organized so that completing them covers those items. Keep the process → structure → properties → performance framing in view as you write: every paragraph should be traceable to that chain.

**What you submit (four files, all on Webcourses):**

1. This notebook printed to PDF (`EMA-6611-HW02-LastName.pdf`) — the graded record of your analysis.
2. The notebook file (`.ipynb`).
3. Your extended `survey.csv` (Task 2.4 and Task 3.1 add columns to the file from Homework 1).
4. **Your report compiled from Overleaf as of submission** (`LastName-Report-HW02.pdf`). The report sections named in each task are graded from this PDF; the notebook is the supporting evidence. Template instruction text may remain in sections you have not reached yet, but must be gone from the three sections graded here.

**Grading:** Task 1 — 20 points; Task 2 — 45 points; Task 3 — 35 points; bonus (Task 2.5) — up to 5 points. Each task states how its points divide between the notebook and the report.


## Ground Rules

* Work through the notebook top to bottom. Cells marked **`# YOUR CODE HERE`** and *Your answer here* are graded deliverables; the instructor-provided module cells are run as they are.
* **AI use** (per the course AI policy): you *may* use generative AI — including the Gemini assistant built into Colab — for code (for example, "build the wurtzite GaN structure with the module above and show the (0001) plane"), for debugging, and for finding and summarizing literature. You must be able to explain every line of code you submit. Do **not** use AI to write the prose in this notebook or in your report. Disclose all AI use in the disclosure cell at the end.
* Every number you take from a datasheet, article or database must be cited in the report with `\cite{}` and must appear in `references.bib`. Cite datasheets as `@misc` entries with manufacturer, document title, revision or date, and URL; cite database values by the original measurement they come from (the loader below prints it).
* Discussing concepts with classmates is encouraged; the work you submit must be your own.
* Before submitting, confirm the whole notebook runs cleanly: **Runtime → Restart and run all**.


## Setup

Run the three cells below first. The second installs three packages that are not preinstalled in Colab (about a minute). The setup must be re-run each time the runtime restarts.


In [ ]:
# Imports and plotting configuration — do not modify
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import constants

plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'figure.dpi': 110,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    # course figure style: tick marks inside, mirrored on the top and right axes, larger
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.major.size': 6, 'ytick.major.size': 6,
    'xtick.minor.size': 3.5, 'ytick.minor.size': 3.5,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
})

# Physical constants (SI unless noted)
q   = constants.e            # elementary charge [C]
kB  = constants.k            # Boltzmann constant [J/K]
kB_eV = constants.k / constants.e   # Boltzmann constant [eV/K]
h   = constants.h            # Planck constant [J·s]
c   = constants.c            # speed of light [m/s]
eps0 = constants.epsilon_0   # vacuum permittivity [F/m]

print('Setup complete.')


In [ ]:
# Package installation (pymatgen: crystal structures; py3Dmol: 3D viewer; rdkit: molecules) — do not modify
%pip -q install pymatgen py3Dmol rdkit
import pymatgen, py3Dmol, rdkit
print('Packages installed.')


In [ ]:
# Upload your survey.csv from Homework 1 to this session (folder icon at left → upload), then run this cell.
import os
assert os.path.exists('survey.csv'), 'survey.csv not found: upload the file from Homework 1 to this Colab session.'
survey = pd.read_csv('survey.csv')
print(f'survey.csv: {len(survey)} articles, {int((survey.top20.astype(str).str.lower()=="yes").sum())} in the top 20.')
survey.head(3)


---
## Task 1 — Commercially Available Devices section of the report (20 points)

Complete the **Commercially Available Devices** section of your report in Overleaf. Points: 15 for the report section, 5 for the notebook answer below.

**1.1 Datasheets and table.** Use at least three datasheets — those from Homework 1, or better ones you have found since (the same "at most two from one manufacturer" rule applies; two datasheets are acceptable for topics marked \* on the topic menu). Fill in the commercial-device table file in the template that matches your device type (`table-commercial-pv-cells.tex`, `-leds.tex` or `-photodetectors.tex`): one row per product, values from the datasheets only, every value cited. Keep the columns as they are; add a column only if your device class has a figure of merit the template omits, and say so in the caption.

**1.2 Annotated device figure.** Include one figure showing the device or module construction (a datasheet drawing, a cross-section, or a photograph you annotate yourself) with the source acknowledged in the caption, as the template examples show. Label the layers or components you will discuss in the Discussion section — this figure is the reader's map for the rest of the report.

**1.3 Narrative.** Write the section as a comparison, not three product descriptions in a row. Cover, to the extent each datasheet allows: the company and the product's intended application; the device design and bill of materials; the performance parameters and the test conditions under which they are quoted; and the durability information — warranty terms for modules, lifetime ratings (L70/L50, MTTF) for LEDs and detectors. Then address transparency directly: what does each manufacturer disclose, what do they leave out, and which missing items would matter most to an engineer selecting the product? Close with one paragraph that connects the products to the analysis to come — for example, which building blocks (absorber, contacts, passivation, encapsulation) the datasheets say nothing about, so that the literature will have to fill the gap.

**1.4 Bibliography.** Each datasheet is a `@misc` entry in `references.bib`, cited where it is used (table rows, figure caption, text). Compile and check that no `[?]` remains.


**Task 1 Answer** *(5 points; the section itself is graded from your report PDF)*

**Datasheets used** (manufacturer, model, datasheet date/revision, BibTeX key, link):

1. *(...)*
2. *(...)*
3. *(...)*

**Least-disclosed item across the three datasheets, and why it matters:** *(two or three sentences)*

**Report status checklist:**

* ☐ Table complete
* ☐ Figure included with its source acknowledged
* ☐ Narrative written as a comparison
* ☐ All datasheets cited and the project compiles without `[?]`


---
## Task 2 — Absorbers or Emitters subsection of the report (45 points + 5 bonus)

Complete the **Absorbers** or **Emitters** subsection of the Discussion and Analysis in Overleaf (retitle it to match your device). Points: 2.1 — 8 (report); 2.2 — 10 (5 notebook, 5 report); 2.3 — 12 (6 notebook, 6 report); 2.4 — 15 (5 notebook validation table, 10 report narrative); 2.5 — bonus 5.

### 2.1 Crystal structure: lattice and basis (8 points, report)

Identify the crystal structure of your absorber or emitter and describe it in the report as a *lattice plus a basis*: the Bravais lattice and space group, the lattice constant(s) with their source, and the atoms of the basis with their fractional coordinates. When the basis contains more than one atom, each basis atom generates its own copy of the lattice, and that copy is called a **sublattice**: in zincblende GaAs the Ga atoms form one face-centered-cubic sublattice and the As atoms form another, displaced from the first by one quarter of the body diagonal; in wurtzite GaN the Ga and N sublattices are two interpenetrating hexagonal-close-packed arrays; in the perovskite ABX$_3$ the A cations, the B cations and the X anions occupy three distinct sublattices. State which sublattice each element in your material occupies. For an alloy (InGaN, AlGaInP, CIGS, HgCdTe, ...), say which sublattice the alloying element substitutes on — In replaces Ga on the group-III sublattice of InGaN while the N sublattice is untouched — and give the composition range used in your device. For an organic or molecular material, replace this with the molecular structure, the way the molecules pack in the film (amorphous, semicrystalline, liquid-crystalline), and what that packing does to charge transport.

Cite a materials-data source for the lattice constants (for example Vurgaftman, Meyer & Ram-Mohan, J. Appl. Phys. 89, 5815 (2001) for III–Vs; Vurgaftman & Meyer, J. Appl. Phys. 94, 3675 (2003) for nitrides; Madelung, *Semiconductors: Data Handbook* (Springer, 2004); or the Materials Project entry, Jain et al., APL Mater. 1, 011002 (2013)).

### 2.2 Unit cell, lattice planes and planar densities (10 points)

The module below builds a conventional unit cell with **pymatgen** (Ong et al., Comput. Mater. Sci. 68, 314 (2013)) from the space group, lattice constants and basis, renders it with **py3Dmol** (Rego & Koes, Bioinformatics 31, 2322 (2015)) with each element in its own color, projects the atoms onto a chosen lattice plane, and computes the atomic planar density of that plane. A table of the materials on the topic menu is built in; if yours is missing, add an entry with a citation or ask Gemini to write the `Structure.from_spacegroup(...)` call for it.

In the code cells that follow the module, produce for your material:

(a) **The 3D structure** as a 2 × 2 × 2 supercell (the default), so that the familiar sub-arrangements are visible — the face-centered-cubic frame inside the diamond and zincblende structures, the ABAB stacking of wurtzite, the corner-sharing octahedra of the perovskite. Rotate it until one of these is recognizable, take a screenshot (the interactive viewer does not survive printing), and describe in the caption what the reader should see. Add `labels=True` if you want the element symbols drawn on the atoms.

(b) **2D views of the three principal planes** — (100), (110), (111) for cubic structures; (0001), (10-10), (11-20) for wurtzite (enter them as (0 0 1), (1 0 0), (1 1 0)); the chalcopyrite (112) plane corresponds to the zincblende (111). The plots show two successive atomic layers along the plane normal (large symbols: top layer; faint: the layer beneath).

(c) **The atomic planar density of each plane**, the number of atoms per unit area on one atomic layer. First calculate it *by hand* from the lattice constant and the arrangement you see in (b) — for example, the (100) layer of a diamond-cubic crystal holds 2 atoms per $a^2$ — and then check your numbers with `planar_density`, which computes them exactly from a surface-oriented cell. Note where the layers come in closely spaced pairs (the (111) bilayer of diamond and zincblende, the cation–anion bilayer of wurtzite (0001)) and, for compounds, which planes are polar (all cations or all anions) and which are mixed. Put the figures in the report with a caption that names the structure, the source of the lattice constants, the tool, and the planar densities.

Then answer, in the report: **on which crystallographic orientation is your device grown or cut, and why?** (Substrate orientation and polarity for epitaxial films; wafer orientation and texturing for silicon — the (111) planes exposed by alkaline texturing are the densest ones you just computed; grain orientation or texture for polycrystalline thin films; not applicable for amorphous and molecular films — say so.) Your corpus papers state this in their experimental sections.

**Organic or molecular materials:** use `show_molecule('<name or SMILES>')` (RDKit; PubChem lookup by name) for the emitter, host, or transport molecule and the repeat unit of a polymer, and describe the film morphology from your corpus instead of lattice planes and densities.


In [ ]:
# ============================================================
# HW02 MODULE A — crystal structures (instructor-provided; run, do not modify)
# Builds a unit cell with pymatgen, renders it with py3Dmol, and projects the
# atoms onto a chosen (hkl) plane. Organic/molecular fallback uses RDKit + PubChem.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from pymatgen.core import Lattice, Structure
from pymatgen.io.cif import CifWriter
from pymatgen.core.surface import SlabGenerator
import py3Dmol

# Lattice constants (Å) at 300 K. Sources: Vurgaftman, Meyer & Ram-Mohan, J. Appl. Phys. 89, 5815 (2001)
# for III–V; Vurgaftman & Meyer, J. Appl. Phys. 94, 3675 (2003) for nitrides; Madelung, Semiconductors:
# Data Handbook (Springer, 2004) for the rest. Edit or add entries, citing your own source.
CRYSTALS = {
    # --- diamond (Fd-3m): one species on the fcc lattice with a two-atom basis (0,0,0),(1/4,1/4,1/4)
    'Si':   dict(sg='Fd-3m', a=5.431, species=['Si'], coords=[[0, 0, 0]], structure='diamond cubic'),
    'Ge':   dict(sg='Fd-3m', a=5.658, species=['Ge'], coords=[[0, 0, 0]], structure='diamond cubic'),
    # --- zincblende (F-43m): cation at (0,0,0), anion at (1/4,1/4,1/4)
    'GaAs': dict(sg='F-43m', a=5.653, species=['Ga', 'As'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'AlAs': dict(sg='F-43m', a=5.661, species=['Al', 'As'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'InP':  dict(sg='F-43m', a=5.869, species=['In', 'P'],  coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'GaP':  dict(sg='F-43m', a=5.451, species=['Ga', 'P'],  coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'InAs': dict(sg='F-43m', a=6.058, species=['In', 'As'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'GaSb': dict(sg='F-43m', a=6.096, species=['Ga', 'Sb'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'InSb': dict(sg='F-43m', a=6.479, species=['In', 'Sb'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'CdTe': dict(sg='F-43m', a=6.482, species=['Cd', 'Te'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'HgTe': dict(sg='F-43m', a=6.461, species=['Hg', 'Te'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    'ZnSe': dict(sg='F-43m', a=5.668, species=['Zn', 'Se'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='zincblende'),
    # --- wurtzite (P6_3mc): cation at (1/3,2/3,0), anion at (1/3,2/3,u)
    'GaN':  dict(sg='P6_3mc', a=3.189, c=5.185, species=['Ga', 'N'], coords=[[1/3, 2/3, 0], [1/3, 2/3, 0.377]], structure='wurtzite'),
    'AlN':  dict(sg='P6_3mc', a=3.112, c=4.982, species=['Al', 'N'], coords=[[1/3, 2/3, 0], [1/3, 2/3, 0.382]], structure='wurtzite'),
    'InN':  dict(sg='P6_3mc', a=3.545, c=5.703, species=['In', 'N'], coords=[[1/3, 2/3, 0], [1/3, 2/3, 0.377]], structure='wurtzite'),
    'ZnO':  dict(sg='P6_3mc', a=3.250, c=5.207, species=['Zn', 'O'], coords=[[1/3, 2/3, 0], [1/3, 2/3, 0.382]], structure='wurtzite'),
    'CdS':  dict(sg='P6_3mc', a=4.136, c=6.714, species=['Cd', 'S'], coords=[[1/3, 2/3, 0], [1/3, 2/3, 0.377]], structure='wurtzite'),
    # --- rock salt (Fm-3m)
    'PbS':  dict(sg='Fm-3m', a=5.936, species=['Pb', 'S'], coords=[[0, 0, 0], [0.5, 0.5, 0.5]], structure='rock salt'),
    # --- chalcopyrite (I-42d): Cu 4a (0,0,0), In 4b (0,0,1/2), Se 8d (x,1/4,1/8), x ≈ 0.224
    'CuInSe2': dict(sg='I-42d', a=5.782, c=11.62, species=['Cu', 'In', 'Se'],
                    coords=[[0, 0, 0], [0, 0, 0.5], [0.224, 0.25, 0.125]], structure='chalcopyrite'),
    'CuGaSe2': dict(sg='I-42d', a=5.614, c=11.03, species=['Cu', 'Ga', 'Se'],
                    coords=[[0, 0, 0], [0, 0, 0.5], [0.250, 0.25, 0.125]], structure='chalcopyrite'),
    # --- cuprite (Pn-3m): O at 2a (0,0,0) [origin choice 1], Cu at 4b (1/4,1/4,1/4)
    'Cu2O': dict(sg='Pn-3m', a=4.270, species=['O', 'Cu'], coords=[[0, 0, 0], [0.25, 0.25, 0.25]], structure='cuprite'),
    # --- cubic perovskite ABX3 (Pm-3m): B at (0,0,0), X at (1/2,0,0), A at (1/2,1/2,1/2)
    'CsPbBr3': dict(sg='Pm-3m', a=5.874, species=['Pb', 'Br', 'Cs'], coords=[[0, 0, 0], [0.5, 0, 0], [0.5, 0.5, 0.5]], structure='cubic perovskite'),
    'CsPbI3':  dict(sg='Pm-3m', a=6.289, species=['Pb', 'I', 'Cs'],  coords=[[0, 0, 0], [0.5, 0, 0], [0.5, 0.5, 0.5]], structure='cubic perovskite (high-T alpha phase)'),
    # MAPbI3: the methylammonium cation is shown as a single placeholder atom (N) on the A site of the pseudo-cubic cell
    'MAPbI3':  dict(sg='Pm-3m', a=6.28, species=['Pb', 'I', 'N'], coords=[[0, 0, 0], [0.5, 0, 0], [0.5, 0.5, 0.5]], structure='pseudo-cubic perovskite (MA+ shown as one placeholder atom)'),
}

# Alloys: build the parent binary and state the composition; the lattice constant follows Vegard's law.
ALLOY_PARENTS = {'InGaN': 'GaN', 'AlGaN': 'GaN', 'AlGaAs': 'GaAs', 'AlGaInP': 'GaAs', 'GaInP': 'GaAs',
                 'InGaAs': 'InP', 'InGaAsP': 'InP', 'HgCdTe': 'CdTe', 'CIGS': 'CuInSe2', 'InAsSb': 'InAs'}


def build_structure(name, **override):
    """Return a pymatgen Structure for a material in CRYSTALS (or an alloy parent).
    Pass a=..., c=... to override lattice constants (e.g., a Vegard-law value for an alloy)."""
    key = ALLOY_PARENTS.get(name, name)
    if key not in CRYSTALS:
        raise KeyError(f"'{name}' is not in CRYSTALS. Add an entry (space group, lattice constants, basis) with a citation, "
                       f"or build the Structure yourself with pymatgen. Available: {sorted(CRYSTALS)}")
    d = dict(CRYSTALS[key]); d.update(override)
    a = d['a']; c = d.get('c', None)
    if d['sg'].startswith('P6'):
        lat = Lattice.hexagonal(a, c)
    elif d['sg'] == 'I-42d':
        lat = Lattice.tetragonal(a, c)
    else:
        lat = Lattice.cubic(a)
    s = Structure.from_spacegroup(d['sg'], lat, d['species'], d['coords'])
    s.hw_info = dict(name=name, parent=key, **{k: v for k, v in d.items() if k in ('sg', 'a', 'c', 'structure')})
    return s


def describe(structure):
    """Print lattice, basis and density information for the unit cell."""
    info = getattr(structure, 'hw_info', {})
    print(f"{info.get('name', '')}: {info.get('structure', '')}, space group {info.get('sg', '')}")
    print(f"Conventional cell: a = {structure.lattice.a:.3f} Å, b = {structure.lattice.b:.3f} Å, c = {structure.lattice.c:.3f} Å; "
          f"{len(structure)} atoms; volume {structure.volume:.2f} Å^3; density {structure.density:.3f} g/cm^3")
    print(f"Formula (cell): {structure.composition.formula}; atomic density {len(structure)/structure.volume*1e24:.3e} atoms/cm^3")


def show_3d(structure, supercell=(2, 2, 2), width=560, height=460, style='ballstick', labels=False):
    """Interactive 3D view (rotate with the mouse). Atoms are colored by element (Jmol color convention:
    Si tan, Ga/In/Cd/Zn grey-blue tones, As/Te/Se/N/O distinct); labels=True prints the element on each atom.
    A 2x2x2 supercell is the default so that the familiar sub-arrangements (e.g., the fcc frame of the diamond
    structure, the hexagonal stacking of wurtzite) are visible. Works in Colab, not in a printed PDF —
    take a screenshot for your report and cite the structure source."""
    s = structure.copy(); s.make_supercell(supercell)
    cif = str(CifWriter(s))
    v = py3Dmol.view(width=width, height=height)
    v.addModel(cif, 'cif')
    if style == 'ballstick':
        v.setStyle({'sphere': {'scale': 0.28, 'colorscheme': 'Jmol'}, 'stick': {'radius': 0.12, 'colorscheme': 'Jmol'}})
    else:
        v.setStyle({'sphere': {'scale': 0.5, 'colorscheme': 'Jmol'}})
    if labels:
        v.addPropertyLabels('elem', {}, {'fontSize': 11, 'fontColor': 'black', 'showBackground': False, 'alignment': 'center'})
    v.addUnitCell()
    v.zoomTo()
    species = sorted({str(sp) for sp in s.species})
    print(f"Supercell {supercell}: {len(s)} atoms. Species shown: {', '.join(species)} (each element its own color).")
    return v


def planar_density(structure, hkl, tol=0.15, verbose=True):
    """Atomic planar density of the (hkl) atomic layers: atoms per unit area on ONE atomic layer, in cm^-2,
    computed exactly from a surface-oriented cell (pymatgen SlabGenerator). Layers that come in closely spaced pairs
    (the (111) bilayer of diamond/zincblende, the Ga–N bilayer of wurtzite (0001)) are reported separately with
    their spacing. Returns a list of (density_cm2, composition, spacing_to_next_layer_A)."""
    sg = SlabGenerator(structure, miller_index=tuple(int(x) for x in hkl), min_slab_size=15, min_vacuum_size=8,
                       center_slab=True, primitive=False)
    slab = sg.get_slab().get_orthogonal_c_slab()
    A = slab.surface_area                                   # in-plane repeat area, Å^2
    order = np.argsort(slab.cart_coords[:, 2]); z = slab.cart_coords[order, 2]
    sp = np.array([str(x) for x in slab.species])[order]
    groups = [[0]]
    for i in range(1, len(z)):
        if z[i] - z[groups[-1][-1]] > tol: groups.append([i])
        else: groups[-1].append(i)
    layers = []
    for g in groups:
        comp = {s: int((sp[g] == s).sum()) for s in sorted(set(sp[g]))}
        layers.append([float(z[g].mean()), len(g) / A * 1e16, comp])
    inner = layers[2:-2] if len(layers) > 6 else layers   # drop the slab surfaces
    seen, out = set(), []
    for i, (zc, dens, comp) in enumerate(inner[:-1]):
        key = (round(dens, -11), tuple(comp.items()), round(inner[i + 1][0] - zc, 2))
        if key in seen: continue
        seen.add(key); out.append((dens, comp, inner[i + 1][0] - zc))
    if verbose:
        h = ' '.join(str(int(x)) for x in hkl)
        print(f"({h}) planes: in-plane repeat area {A:.2f} Å^2")
        for dens, comp, spacing in out:
            cs = ' + '.join(f'{n} {s}' for s, n in comp.items())
            print(f"   layer of {cs} per repeat area -> {dens:.3e} atoms/cm^2; next layer {spacing:.2f} Å away")
    return out


def plane_view(structure, hkl, n_planes=2, size=3, ax=None, title=None):
    """Project the atoms lying on the first `n_planes` lattice planes (hkl) onto that plane.
    Use 3-index (h k l) for cubic/tetragonal cells; for hexagonal cells give (h k l) with the
    4-index i implied, e.g. (0 0 1) for the basal plane (0001), (1 0 0) for (10-10), (1 1 0) for (11-20).
    Successive atomic layers along the plane normal are drawn large (top layer) and faint (layer beneath).
    Returns the spacing d_hkl of the conventional-cell planes in Å."""
    hkl = np.array(hkl, dtype=float)
    lat = structure.lattice
    G = lat.reciprocal_lattice_crystallographic.get_cartesian_coords(hkl)   # plane normal, |G| = 1/d
    d = 1.0 / np.linalg.norm(G); nhat = G / np.linalg.norm(G)
    # in-plane orthonormal basis
    trial = np.array([1.0, 0, 0]) if abs(nhat[0]) < 0.9 else np.array([0, 1.0, 0])
    u = np.cross(nhat, trial); u /= np.linalg.norm(u); v = np.cross(nhat, u)
    s = structure.copy(); s.make_supercell([size, size, size])
    cart = s.cart_coords - s.cart_coords.mean(axis=0)
    heights = cart @ nhat
    # distinct plane heights (rounded), pick n_planes nearest the middle
    levels = np.unique(np.round(heights / d * 1e3) / 1e3 * d)
    mid = levels[np.argsort(abs(levels - np.median(levels)))][:n_planes]
    mid = np.sort(mid)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.2, 5.2))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    species = sorted({str(sp) for sp in s.species})
    cmap = {sp: colors[i % len(colors)] for i, sp in enumerate(species)}
    for j, lv in enumerate(mid):
        sel = abs(heights - lv) < 0.02 * d + 1e-3
        pts = cart[sel]; sp = np.array([str(x) for x in s.species])[sel]
        for name in species:
            m = sp == name
            if m.any():
                ax.scatter(pts[m] @ u, pts[m] @ v, s=260 if j == len(mid) - 1 else 110, color=cmap[name],
                           alpha=1.0 if j == len(mid) - 1 else 0.35, edgecolor='k', linewidth=0.5,
                           label=f'{name} (layer {j+1})')
    ax.set_aspect('equal'); ax.set_xlabel('In-Plane x (Å)'); ax.set_ylabel('In-Plane y (Å)')
    ax.set_title(title or f"({' '.join(str(int(x)) for x in hkl)}) plane — d = {d:.3f} Å", fontsize=10)
    ax.legend(fontsize=7, loc='upper right', framealpha=0.9)
    lim = 1.3 * max(structure.lattice.a, structure.lattice.c)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    return d


def plane_views(structure, planes, n_planes=2):
    """Side-by-side plane projections for a list of (hkl) tuples."""
    fig, axs = plt.subplots(1, len(planes), figsize=(5.0 * len(planes), 5.0))
    axs = np.atleast_1d(axs)
    for ax, hkl in zip(axs, planes):
        d = plane_view(structure, hkl, n_planes=n_planes, ax=ax)
    plt.tight_layout()
    return fig


# ---------- molecular fallback (organic emitters / absorbers) ----------
def molecule_from_name_or_smiles(text):
    """Return an RDKit Mol from a SMILES string, or look the name up on PubChem (no API key needed)."""
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.error')          # silence the parse error when the text is a name, not a SMILES
    m = Chem.MolFromSmiles(text)
    RDLogger.EnableLog('rdApp.error')
    if m is not None:
        return m, text
    import urllib.request, urllib.parse, json
    url = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/'
           + urllib.parse.quote(text) + '/property/IsomericSMILES,MolecularFormula,IUPACName/JSON')
    data = json.loads(urllib.request.urlopen(url, timeout=30).read().decode())
    prop = data['PropertyTable']['Properties'][0]
    smi = prop.get('IsomericSMILES') or prop.get('ConnectivitySMILES') or prop.get('SMILES')
    print(f"PubChem: {text} -> {prop.get('MolecularFormula')} ; SMILES = {smi}")
    return Chem.MolFromSmiles(smi), smi


def show_molecule(text, size=(450, 350)):
    """2D depiction of a molecule (organic emitter, transport material, polymer repeat unit) from a name or SMILES."""
    from rdkit.Chem import Draw, Descriptors
    m, smi = molecule_from_name_or_smiles(text)
    print(f"Formula {Descriptors.rdMolDescriptors.CalcMolFormula(m)}, MW = {Descriptors.MolWt(m):.1f} g/mol, "
          f"{m.GetNumHeavyAtoms()} heavy atoms")
    return Draw.MolToImage(m, size=size)


def show_molecule_3d(text, width=450, height=380):
    """3D conformer (RDKit ETKDG) rendered with py3Dmol."""
    from rdkit import Chem
    from rdkit.Chem import AllChem
    m, smi = molecule_from_name_or_smiles(text)
    mh = Chem.AddHs(m); AllChem.EmbedMolecule(mh, randomSeed=1); AllChem.MMFFOptimizeMolecule(mh)
    v = py3Dmol.view(width=width, height=height)
    v.addModel(Chem.MolToMolBlock(mh), 'mol'); v.setStyle({'stick': {}}); v.zoomTo()
    return v

print('Module A loaded: build_structure, describe, show_3d, plane_view(s), planar_density, show_molecule(_3d). Materials:', ', '.join(CRYSTALS))


In [ ]:
# YOUR CODE HERE (Task 2.2a) — replace 'CdTe' with your material.
# Alloys: build the parent binary, e.g. build_structure('InGaN'); pass a=..., c=... to use your own lattice constants.
s = build_structure('CdTe')
describe(s)
show_3d(s, supercell=(2, 2, 2), labels=False)     # rotate with the mouse; screenshot for the report


In [ ]:
# YOUR CODE HERE (Task 2.2b, c) — principal planes and their planar densities for your structure
planes = [(1, 0, 0), (1, 1, 0), (1, 1, 1)]                   # cubic; use [(0,0,1),(1,0,0),(1,1,0)] for wurtzite
fig = plane_views(s, planes)
fig.savefig('planes.png', dpi=200, bbox_inches='tight')     # download from the Files panel for Overleaf
for hkl in planes:
    planar_density(s, hkl)                                  # compare with your hand calculation

# Organic/molecular materials only:
# show_molecule('tris(8-hydroxyquinolinato)aluminium')     # or a SMILES string


**Task 2.2 Answer** *(5 points)*

**Material, structure, space group, lattice constants and their source:** *(...)*

**Planar densities by hand vs. `planar_density` (atoms/cm², one layer each), and which planes are polar:**

| plane | hand calculation (show the arithmetic) | code | polar / mixed |
|---|---|---|---|
| ... | ... | ... | ... |

**Growth or cut orientation of the device and the reason given in your corpus (cite):** *(...)*

**Report status checklist:**

* ☐ Unit-cell figure
* ☐ Plane figure with planar densities in the caption
* ☐ Captions with sources
* ☐ Growth or cut orientation stated and cited


### 2.3 Bandgap and absorption (12 points)

**Absorption coefficient from optical constants.** The module below loads tabulated $n$ and $k$ from the refractiveindex.info database (Polyanskiy, Sci. Data 11, 94 (2024); public domain), which archives published ellipsometry and transmission measurements and prints the original reference for the data set — cite *that* measurement in your report, not the website. The absorption coefficient follows from the extinction coefficient,

$$\alpha(\lambda) = \frac{4\pi\,k(\lambda)}{\lambda},$$

and the depth over which light is absorbed is $1/\alpha$. Materials not in the database (InGaN, AlGaN, CIGS, a-Si:H, HgCdTe and most alloys) must be loaded from your own CSV: take $n,k$ from an ellipsometry paper in your corpus (often in the supplementary information) or from the PV Lighthouse refractive-index library (free account), and cite that source.

In the report, give the bandgap value(s) and type (direct or indirect; for alloys, the composition dependence and the direct–indirect crossover if there is one) from your literature references, with citations. Then present the $\alpha(\lambda)$ figure and explain three features of it: (i) where the absorption edge sits relative to $E_g$; (ii) how steep the edge is — a direct gap rises by orders of magnitude within a few tenths of an eV, an indirect gap rises slowly because a phonon must participate, and a disordered material shows an exponential Urbach tail below the gap; (iii) the magnitude of $\alpha$ above the edge, $10^4$–$10^5$ cm$^{-1}$ for direct-gap materials against $10^2$–$10^3$ cm$^{-1}$ just above the silicon gap. The Tauc construction (Tauc, Grigorovici & Vancu, Phys. Status Solidi 15, 627 (1966)), $(\alpha h\nu)^{1/r}$ against $h\nu$ with $r = 1/2$ for direct and $r = 2$ for indirect allowed transitions, gives an optical gap from the data — compare it with the literature value and comment on the difference. Note that the flat, low-$\alpha$ region below the gap in ellipsometry data is usually the instrument's detection floor, not the material.

**Connect the property to the device (required).** Compute the absorption depth $1/\alpha$ at a wavelength that matters for your device — the peak of the solar spectrum or the band edge for a solar cell, the detection wavelength for a photodetector — and compare it with the absorber thickness reported in your top-20 articles. That one comparison explains why a CdTe absorber is a few micrometres and a silicon wafer is 150 µm, and why a thin absorber needs light trapping or a back reflector.

**Emitters (LEDs): the emission-side version.** Plot $\alpha(\lambda)$ for your emitter material as above and overlay the peak emission wavelength and FWHM from your datasheets (`overlay_emission`). Explain why emission sits at or just below the absorption edge and what self-absorption implies for light extraction. For alloy emitters, use `composition_for_wavelength` to compute the composition that the datasheet peak implies from $E_g(x)$ with the bowing parameter (Vurgaftman et al. 2001; Vurgaftman & Meyer 2003), and compare it with the composition your corpus reports for the active region; the difference is quantum confinement, strain and the choice of parameters — discuss it.


In [ ]:
# ============================================================
# HW02 MODULE B — optical constants and absorption (instructor-provided; run, do not modify)
# Loads n,k from the refractiveindex.info database (Polyanskiy, Sci. Data 11, 94 (2024); CC0 public domain),
# computes alpha = 4*pi*k/lambda, and provides absorption-depth, Tauc and emitter-overlay helpers.
# ============================================================
import io, urllib.request
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from scipy.optimize import brentq

RII_BASE = 'https://raw.githubusercontent.com/polyanskiy/refractiveindex.info-database/master/database/data/'
HC_EV_NM = 1239.84193   # h*c in eV*nm

# Database paths verified September 2026. To use another data set, open the material page on
# refractiveindex.info, click "Data [YAML]" and paste the path after '.../database/data/'.
NK_SOURCES = {
    'Si':        ('main/Si/nk/Green-2008.yml',          'Green, Sol. Energy Mater. Sol. Cells 92, 1305 (2008); 0.25–1.45 µm'),
    'Ge':        ('main/Ge/nk/Nunley.yml',              'Nunley et al., J. Vac. Sci. Technol. B 34, 061205 (2016)'),
    'GaAs':      ('main/GaAs/nk/Papatryfonos.yml',      'Papatryfonos et al., AIP Adv. 11, 025327 (2021); 0.26–1.88 µm'),
    'InP':       ('main/InP/nk/Aspnes.yml',             'Aspnes & Studna, Phys. Rev. B 27, 985 (1983); 0.21–0.83 µm'),
    'GaP':       ('main/GaP/nk/Aspnes.yml',             'Aspnes & Studna 1983'),
    'InAs':      ('main/InAs/nk/Adachi.yml',            'Adachi, J. Appl. Phys. 66, 6030 (1989) (model); 0.21–12.4 µm'),
    'GaSb':      ('main/GaSb/nk/Adachi.yml',            'Adachi 1989 (model); 0.21–12.4 µm'),
    'InSb':      ('main/InSb/nk/Adachi.yml',            'Adachi 1989 (model)'),
    'GaN':       ('main/GaN/nk/Kawashima.yml',          'Kawashima et al., J. Appl. Phys. 82, 3528 (1997); thin film'),
    'AlN':       ('main/AlN/nk/Beliaev1.yml',           'Beliaev et al. 2021; sputtered thin film (check stoichiometry note on the site)'),
    'CdTe':      ('main/CdTe/nk/Treharne.yml',          'Treharne et al., J. Phys.: Conf. Ser. 286, 012038 (2011); 0.3–1.5 µm'),
    'CdS':       ('main/CdS/nk/Treharne.yml',           'Treharne et al. 2011'),
    'ZnSe':      ('main/ZnSe/nk/Adachi.yml',            'Adachi (model)'),
    'AlGaAs (x=0.22)': ('other/semiconductor alloys/AlAs-GaAs/nk/Papatryfonos-21.9.yml', 'Papatryfonos et al. 2021; other compositions on the site'),
    'AlGaAs (x=0.41)': ('other/semiconductor alloys/AlAs-GaAs/nk/Papatryfonos-41.1.yml', 'Papatryfonos et al. 2021'),
    'GaInP':     ('other/semiconductor alloys/GaP-InP/nk/Schubert.yml', 'Schubert et al., J. Appl. Phys. 77, 3416 (1995); Ga0.5In0.5P'),
    'InGaAs':    ('other/semiconductor alloys/GaAs-InAs/nk/Adachi.yml', 'Adachi 1989 (model); In0.53Ga0.47As'),
    'MAPbI3':    ('other/perovskite/CH3NH3PbI3/nk/Phillips.yml', 'Phillips et al., Sol. Energy Mater. Sol. Cells 147, 327 (2016)'),
    'CsPbBr3':   ('other/perovskite/CsPbBr3/nk/Brennan.yml',     'Brennan et al. 2019'),
    'CsPbI3':    ('other/perovskite/CsPbI3/nk/Aftenieva.yml',    'Aftenieva et al.'),
    'P3HT:PCBM': ('other/mixed organic/P3HT-PC61BM/nk/Stelling.yml', 'Stelling et al., Sci. Rep. 7, 42530 (2017)'),
    'PTB7':      ('organic/(C41H53FO4S4)n - PTB7/nk/Kamptner-o.yml', 'Kamptner et al. (ordinary index)'),
    'ITO':       ('other/mixed crystals/In2O3-SnO2/nk/Konig.yml',  'König et al., ACS Nano 8, 6182 (2014)'),
    'AZO':       ('other/doped crystals/Al-ZnO/nk/Treharne.yml',   'Treharne et al. 2011'),
    'TiO2 (film)': ('main/TiO2/nk/Sarkar.yml',          'Sarkar et al. 2019; thin film'),
    'MoO3':      ('main/MoO3/nk/Vos.yml',               'Vos et al.'),
}
NOT_IN_DATABASE = ('InGaN, AlGaN, CIGS (CuIn1-xGaxSe2), a-Si:H, HgCdTe and most alloys are NOT in refractiveindex.info. '
                   'Take n,k from a paper in your corpus (ellipsometry data are often in the SI) or from the PV Lighthouse '
                   'refractive-index library (free account), save two or three columns (wavelength_nm, n, k) as a CSV, '
                   'and load it with load_nk_csv().')


def _fetch(path):
    return urllib.request.urlopen(RII_BASE + urllib.parse.quote(path), timeout=60).read().decode()

import urllib.parse

def load_nk(material_or_path):
    """Return a DataFrame [wavelength_nm, n, k] from refractiveindex.info.
    Accepts a key of NK_SOURCES or a database path such as 'main/Si/nk/Green-2008.yml'."""
    path, label = NK_SOURCES.get(material_or_path, (material_or_path, material_or_path))
    doc = yaml.safe_load(_fetch(path))
    frames = []
    for block in doc['DATA']:
        typ = block.get('type', '')
        if typ.startswith('tabulated'):
            arr = np.loadtxt(io.StringIO(block['data']))
            arr = np.atleast_2d(arr)
            df = pd.DataFrame({'wavelength_nm': arr[:, 0] * 1000.0})
            if typ == 'tabulated nk':
                df['n'], df['k'] = arr[:, 1], arr[:, 2]
            elif typ == 'tabulated n':
                df['n'] = arr[:, 1]
            elif typ == 'tabulated k':
                df['k'] = arr[:, 1]
            frames.append(df)
        else:
            print(f"Note: '{path}' stores {typ} (a dispersion formula for n, no k). alpha cannot be computed from it.")
    if not frames:
        raise ValueError('No tabulated data in this file.')
    df = frames[0]
    for other in frames[1:]:      # merge separate n and k tables on wavelength by interpolation
        for col in ('n', 'k'):
            if col in other and col not in df:
                df[col] = np.interp(df['wavelength_nm'], other['wavelength_nm'], other[col])
    if 'k' not in df: df['k'] = np.nan
    if 'n' not in df: df['n'] = np.nan
    df = df.sort_values('wavelength_nm').reset_index(drop=True)
    df.attrs['source'] = label; df.attrs['path'] = path
    df.attrs['reference'] = doc.get('REFERENCES', '').strip()
    print(f"Loaded {path}: {len(df)} points, {df.wavelength_nm.min():.0f}–{df.wavelength_nm.max():.0f} nm. "
          f"Reference: {df.attrs['reference'][:120]}")
    return df


def load_nk_csv(filename, wavelength_unit='nm'):
    """Load your own n,k data (columns: wavelength, n, k) from a CSV; wavelength_unit 'nm' or 'um'."""
    df = pd.read_csv(filename)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = [c for c in df.columns if c.startswith('wave') or c in ('lambda', 'lam', 'wl')][0]
    out = pd.DataFrame({'wavelength_nm': df[wcol] * (1000.0 if wavelength_unit == 'um' else 1.0), 'n': df['n'], 'k': df['k']})
    out.attrs['source'] = filename
    return out.sort_values('wavelength_nm').reset_index(drop=True)


def add_alpha(df):
    """Add photon energy (eV) and absorption coefficient alpha (cm^-1) = 4*pi*k/lambda."""
    df = df.copy()
    df['E_eV'] = HC_EV_NM / df['wavelength_nm']
    df['alpha_cm'] = 4 * np.pi * df['k'] / (df['wavelength_nm'] * 1e-7)
    return df


def plot_absorption(df, Eg=None, label=None, lam_marks=(), ax=None):
    """alpha vs wavelength (left) and vs photon energy (right), log scale; optional Eg line and wavelength marks."""
    df = add_alpha(df); d = df[df.alpha_cm > 0]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    axs[0].semilogy(d.wavelength_nm, d.alpha_cm, lw=1.8, label=label or df.attrs.get('source', ''))
    axs[0].set_xlabel('Wavelength (nm)'); axs[0].set_ylabel(r'Absorption Coefficient $\alpha$ (cm$^{-1}$)')
    axs[1].semilogy(d.E_eV, d.alpha_cm, lw=1.8)
    axs[1].set_xlabel('Photon Energy (eV)'); axs[1].set_ylabel(r'Absorption Coefficient $\alpha$ (cm$^{-1}$)')
    if Eg:
        axs[1].axvline(Eg, color='k', ls='--', lw=1); axs[1].text(Eg, d.alpha_cm.max(), f' $E_g$ = {Eg} eV', va='top', fontsize=9)
        axs[0].axvline(HC_EV_NM / Eg, color='k', ls='--', lw=1)
    for lam in lam_marks:
        axs[0].axvline(lam, color='C1', ls=':', lw=1.2); axs[0].text(lam, d.alpha_cm.min() * 2, f' {lam:.0f} nm', color='C1', fontsize=8, rotation=90)
    for ax in axs: ax.grid(True, which='major', alpha=0.25)
    axs[0].legend(fontsize=8); plt.tight_layout()
    return fig


def absorption_depth(df, wavelength_nm):
    """1/alpha (µm) at a wavelength, and the single-pass absorbed fraction 1-exp(-alpha*t) for a few thicknesses."""
    df = add_alpha(df)
    a = np.interp(wavelength_nm, df.wavelength_nm, df.alpha_cm)
    if not np.isfinite(a) or a <= 0:
        print(f'alpha is not available at {wavelength_nm} nm (outside the data range or k = 0).'); return np.nan
    depth_um = 1e4 / a
    print(f"lambda = {wavelength_nm:.0f} nm: alpha = {a:.3e} cm^-1, absorption depth 1/alpha = {depth_um:.3g} µm")
    for t_um in (0.1, 0.5, 1, 2, 5, 10, 50, 150):
        print(f"   thickness {t_um:>5} µm -> single-pass absorbed fraction 1-exp(-alpha t) = {1-np.exp(-a*t_um*1e-4):.3f}")
    return depth_um


def tauc_plot(df, kind='direct', fit_range=None, ax=None):
    """Tauc plot (Tauc, Grigorovici & Vancu, Phys. Status Solidi 15, 627 (1966)): (alpha*E)^(1/r) vs E with
    r = 1/2 for allowed direct transitions and r = 2 for allowed indirect. A straight-line fit over fit_range=(E1, E2)
    extrapolated to zero gives an optical-gap estimate. Returns Eg_fit (eV)."""
    df = add_alpha(df); d = df[(df.alpha_cm > 0)].sort_values('E_eV')
    r = 0.5 if kind == 'direct' else 2.0
    y = (d.alpha_cm * d.E_eV) ** (1.0 / r); x = d.E_eV.values
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.5, 4.2))
    ax.plot(x, y, lw=1.6, label=f'({chr(945)}E)^{1/r:g}')
    Eg_fit = np.nan
    if fit_range:
        m = (x >= fit_range[0]) & (x <= fit_range[1])
        if m.sum() >= 3:
            p = np.polyfit(x[m], y[m], 1); Eg_fit = -p[1] / p[0]
            xx = np.linspace(Eg_fit, x[m].max(), 50)
            ax.plot(xx, np.polyval(p, xx), 'k--', lw=1, label=f'linear fit -> $E_g$ = {Eg_fit:.2f} eV')
    ax.set_xlabel('Photon Energy (eV)'); ax.set_ylabel(f'(αE)^{1/r:g}')
    if fit_range:            # show the region around the edge, not the whole spectrum
        lo, hi = fit_range[0] - 0.5, fit_range[1] + 0.3
        ax.set_xlim(lo, hi); w = (x >= lo) & (x <= hi)
        if w.any(): ax.set_ylim(0, np.nanmax(y[w]) * 1.05)
    else:
        ax.set_ylim(0, np.nanpercentile(y, 98) * 1.05)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.set_title(f'Tauc Plot, {kind.capitalize()} Transition', fontsize=10)
    return Eg_fit


def overlay_emission(ax, peak_nm, fwhm_nm, label='emission (datasheet)'):
    """Shade the emission band (peak ± FWHM/2) on a wavelength-axis plot — for LED topics."""
    ax.axvspan(peak_nm - fwhm_nm / 2, peak_nm + fwhm_nm / 2, color='C3', alpha=0.15, label=label)
    ax.axvline(peak_nm, color='C3', lw=1.2)
    ax.legend(fontsize=8)


# --- alloy band gaps at 300 K: Eg(x) = (1-x) Eg_A + x Eg_B - b x (1-x).
# Values: Vurgaftman, Meyer & Ram-Mohan, J. Appl. Phys. 89, 5815 (2001); Vurgaftman & Meyer, J. Appl. Phys. 94,
# 3675 (2003) for nitrides. Replace with the values from your own sources and cite them.
ALLOY_GAPS = {
    'InxGa1-xN':   dict(EgA=3.42, EgB=0.78, b=1.4,  A='GaN', B='InN'),
    'AlxGa1-xN':   dict(EgA=3.42, EgB=6.13, b=0.7,  A='GaN', B='AlN'),
    'AlxGa1-xAs':  dict(EgA=1.424, EgB=3.03, b=0.37, A='GaAs', B='AlAs (Gamma valley; direct only for x < ~0.45)'),
    '(AlxGa1-x)0.5In0.5P': dict(EgA=1.91, EgB=2.52, b=0.18, A='Ga0.5In0.5P', B='Al0.5In0.5P (Gamma; direct only for x < ~0.53)'),
    'InxGa1-xAs':  dict(EgA=1.424, EgB=0.354, b=0.477, A='GaAs', B='InAs'),
    'Hg1-xCdxTe':  dict(EgA=-0.15, EgB=1.50, b=0.0, A='HgTe (semimetal)', B='CdTe'),
}

def alloy_gap(alloy, x):
    p = ALLOY_GAPS[alloy]
    return (1 - x) * p['EgA'] + x * p['EgB'] - p['b'] * x * (1 - x)

def composition_for_wavelength(alloy, peak_nm):
    """Solve Eg(x) = hc/lambda for the alloy fraction x that emits at peak_nm (300 K, band-edge emission)."""
    E = HC_EV_NM / peak_nm
    f = lambda x: alloy_gap(alloy, x) - E
    xs = np.linspace(0, 1, 2001); fx = f(xs)
    roots = [brentq(f, xs[i], xs[i + 1]) for i in range(len(xs) - 1) if fx[i] * fx[i + 1] < 0]
    if not roots:
        print(f'{alloy}: no composition gives {peak_nm} nm ({E:.2f} eV) with these parameters.'); return np.nan
    x = roots[0]
    print(f"{alloy}: peak {peak_nm:.0f} nm = {E:.3f} eV  ->  x = {x:.3f}  (Eg({x:.3f}) = {alloy_gap(alloy, x):.3f} eV; "
          f"assumes band-edge emission at 300 K; quantum confinement and strain shift this)")
    return x

print('Module B loaded: load_nk, load_nk_csv, add_alpha, plot_absorption, absorption_depth, tauc_plot, overlay_emission, alloy_gap, composition_for_wavelength')
print('Materials with verified database paths:', ', '.join(NK_SOURCES))
print(NOT_IN_DATABASE)


In [ ]:
# YOUR CODE HERE (Task 2.3) — replace 'CdTe' and the numbers with your material
nk = load_nk('CdTe')                      # or: nk = load_nk_csv('my_nk.csv', wavelength_unit='nm')
Eg_lit = 1.50                             # bandgap from your literature (eV) — cite it in the report
fig = plot_absorption(nk, Eg=Eg_lit, label='CdTe (Treharne et al. 2011)', lam_marks=(600, 800))
fig.savefig('absorption.png', dpi=200, bbox_inches='tight')

depth_um = absorption_depth(nk, 600)      # choose the wavelength that matters for YOUR device


In [ ]:
# YOUR CODE HERE (Task 2.3) — Tauc construction; choose 'direct' or 'indirect' and a fit window just above the edge
fig, ax = plt.subplots(figsize=(5.5, 4.2))
Eg_tauc = tauc_plot(nk, kind='direct', fit_range=(1.55, 1.80), ax=ax)
fig.savefig('tauc.png', dpi=200, bbox_inches='tight')

# Emitters only:
# fig = plot_absorption(nk, Eg=Eg_lit, label='GaN'); overlay_emission(fig.axes[0], peak_nm=450, fwhm_nm=20)
# composition_for_wavelength('InxGa1-xN', 450)


**Task 2.3 Answer** *(6 points)*

**Data set used and its original reference:** *(...)*

**Bandgap value and type from the literature (with citations) vs. the Tauc estimate:** *(...)*

**Absorption depth at ___ nm = ___ µm; absorber thickness in your corpus = ___; what this implies for the device design:** *(three or four sentences)*

**Emitters: composition implied by the datasheet peak vs. the composition in the corpus:** *(...)*

**Report status checklist:**

* ☐ Absorption figure with caption and data source
* ☐ Bandgap paragraph with citations
* ☐ Absorption depth compared with the absorber thickness in the corpus


### 2.4 Processing, microstructure, doping and defects (15 points)

This is the process → structure part of the subsection, and it is where reports most often degenerate into one paragraph per paper. To prevent that, build the evidence as a table first and write from the table.

**Extend `survey.csv`.** Add the columns below and fill them for your **top-20 rows** (other rows may stay empty). Write `not reported` when an article does not give the item — those entries are data too: they are the gaps your narrative must name.

| column | content |
|---|---|
| `growth_method` | how the absorber/emitter was made (e.g., MOCVD, Czochralski + diffusion, close-space sublimation, co-evaporation, spin coating, thermal evaporation) |
| `growth_temperature_C` | growth or anneal temperature, °C |
| `substrate` | substrate material and orientation |
| `microstructure` | single crystal / polycrystalline (grain size) / amorphous / nanocrystalline / molecular film |
| `dopant` | dopant species and type (n/p), or `undoped`, `unintentional` |
| `doping_cm3` | carrier or dopant concentration, cm$^{-3}$ |
| `resistivity_ohmcm` | resistivity, Ω cm (or sheet resistance with units) |
| `dominant_defect` | the defect the article identifies as limiting (dislocations, grain boundaries, point defects, dangling bonds, ion migration, ...) |
| `defect_effect` | the performance quantity it affects (lifetime, $V_{oc}$, EQE, droop, dark current, ...) |

Also add the Task 3.1 columns now (they are validated together below). Keep the master copy in your Drive, upload the extended file to this session, and run the validation cell.

**Write the narrative** (10 points, report) from the validated table: which processes dominate your literature and at what thermal budget; the microstructure they produce and how the papers characterize it (XRD, TEM, EBSD, Raman, PL); the dopants, concentrations and resistivities and how they were measured (Hall, four-point probe, ECV, C–V); and the crystallographic and electronic defects the field identifies as limiting, with their effect on performance. Cite the articles that report each item and say explicitly which items your corpus leaves unreported. Two paragraphs of synthesis beat two pages of summary.


In [ ]:
# ============================================================
# HW02 MODULE D — survey.csv extension and validation (instructor-provided; run, do not modify)
# ============================================================
import pandas as pd, numpy as np

HW1_COLUMNS = ["first_author", "year", "title", "doi", "device_type", "absorber_emitter",
               "key_fom_name", "key_fom_value", "processing_disclosed", "top20", "bibtex_key"]
AE_COLUMNS = ["growth_method", "growth_temperature_C", "substrate", "microstructure",
              "dopant", "doping_cm3", "resistivity_ohmcm", "dominant_defect", "defect_effect"]
CSJ_COLUMNS = ["contact_geometry", "e_contact_stack", "e_contact_route", "h_contact_stack", "h_contact_route",
               "rho_c_mohmcm2", "J0_fAcm2", "other_contact_metric"]
NEW_COLUMNS = AE_COLUMNS + CSJ_COLUMNS
ROUTES = {"doping", "band offsets", "induced carriers", "doping + induced carriers", "band offsets + doping",
          "not reported", "not applicable"}
GEOMETRIES = {"top-bottom", "top-top", "bottom-bottom", "not reported", "not applicable"}


def validate_survey(filename="survey.csv"):
    df = pd.read_csv(filename)
    missing = [c for c in HW1_COLUMNS + NEW_COLUMNS if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"
    top = df[df["top20"].astype(str).str.strip().str.lower() == "yes"].copy()
    assert len(top) == 20, f"Exactly 20 rows must have top20 == yes (found {len(top)})."
    for c in NEW_COLUMNS:
        top[c] = top[c].astype(str).str.strip()
    empties = {c: int((top[c].isin(["", "nan", "NaN"])).sum()) for c in NEW_COLUMNS}
    empties = {k: v for k, v in empties.items() if v}
    assert not empties, f"Empty cells in top-20 rows (write 'not reported' if the article does not give it): {empties}"
    bad_route = sorted(set(top["e_contact_route"]).union(top["h_contact_route"]) - ROUTES)
    assert not bad_route, f"Unrecognized route labels {bad_route}; use one of {sorted(ROUTES)}"
    bad_geom = sorted(set(top["contact_geometry"]) - GEOMETRIES)
    assert not bad_geom, f"Unrecognized contact_geometry {bad_geom}; use one of {sorted(GEOMETRIES)}"
    nr = {c: int((top[c].str.lower() == "not reported").sum()) for c in NEW_COLUMNS}
    print(f"Corpus: {len(df)} articles; top-20 rows checked: {len(top)}")
    print("'not reported' counts among the top 20 (these are the gaps your narrative must name):")
    for c, v in nr.items():
        print(f"   {c:22s} {v:2d} / 20")
    print("\nValidation PASSED. The summaries below are part of your submission:")
    with pd.option_context("display.max_rows", 50, "display.width", 120):
        print("\nGrowth / deposition methods (top 20):"); print(top["growth_method"].value_counts().to_string())
        print("\nMicrostructure (top 20):"); print(top["microstructure"].value_counts().to_string())
        print("\nElectron-contact route (top 20):"); print(top["e_contact_route"].value_counts().to_string())
        print("\nHole-contact route (top 20):"); print(top["h_contact_route"].value_counts().to_string())
        print("\nContact geometry (top 20):"); print(top["contact_geometry"].value_counts().to_string())
    both = top[(top["rho_c_mohmcm2"].str.lower() != "not reported") & (top["J0_fAcm2"].str.lower() != "not reported")]
    print(f"\nArticles reporting BOTH rho_c and J0 (a fully characterized contact): {len(both)} / 20")
    return df

print("Module D loaded: validate_survey(). New columns required for the top-20 rows:", ", ".join(NEW_COLUMNS))


In [ ]:
# Task 2.4 / 3.1 validation — upload the extended survey.csv to this session, then run this cell.
survey = validate_survey('survey.csv')


**Task 2.4 Answer** *(5 points for a passing validation cell with the summary printed above; the narrative is graded from the report)*

**Items most often 'not reported' in your corpus, and the articles that do report them (bibtex keys):** *(...)*

**Report status checklist:**

* ☐ Processing, microstructure, doping and defect narrative written from the table
* ☐ Gaps in the corpus named explicitly


### 2.5 Bonus — the E–k diagram (up to 5 points)

Find or compute a representative electronic band structure $E(\mathbf{k})$ for your crystal and place it in the report with a short paragraph that (i) identifies the conduction-band minimum and valence-band maximum and hence the direct or indirect character, (ii) reads off the gap and compares it with the optical gap of Task 2.3, and (iii) notes the effective-mass or valley features that matter for your device (satellite valleys, heavy/light holes, the split-off band).

Sources, in order of preference: a band structure from a paper in your corpus or from a standard reference (reproduce with the source acknowledged in the caption); the Materials Project entry for your compound (DFT band structure — note in the caption that standard DFT underestimates the gap, typically by 30–50 %, and quote the corrected value if the entry provides one); or, for the cubic III–V, II–VI and group-IV materials, a tight-binding calculation (an sp$^3$s$^*$ script with the parameters of Vogl, Hjalmarson & Dow, J. Phys. Chem. Solids 44, 365 (1983) will be posted on the course repository). Organic and molecular materials have no $E(\mathbf{k})$; substitute the HOMO/LUMO levels and the exciton binding energy with citations.


**Task 2.5 Answer (bonus)**

**Source of the E–k diagram; CBM/VBM locations; gap read from it vs. the optical gap:** *(...)*


---
## Task 3 — Carrier-Selective Junctions subsection of the report (35 points)

Complete the **Carrier-Selective Junctions** subsection in Overleaf. Points: 3.1 — 12 (4 notebook, 8 report); 3.2 — 18 (8 notebook, 10 report); 3.3 — 5 (report).

### What "carrier-selective" means, and the vocabulary to use

A device delivers or accepts power only if electrons leave through one terminal and holes through the other. A region is an **electron-selective contact** when the electron conductivity there greatly exceeds the hole conductivity, $\sigma_n \gg \sigma_p$, so that electrons reach the metal with negligible loss of their quasi-Fermi level while holes carry almost no current; a **hole-selective contact** is the mirror image (Würfel, Cuevas & Würfel, IEEE J. Photovolt. 5, 461 (2015)). A built-in electric field is neither necessary nor sufficient for this. Since $\sigma_n/\sigma_p = n\mu_n/p\mu_p$, selectivity is a carrier-density asymmetry, and there are three ways to make one — use these labels in your table and narrative:

* **doping** — a $pn$ homojunction: $n/p = N_D^2/n_i^2$ in an $n$-type region (diffused or implanted emitters, epitaxially doped layers);
* **band offsets** — a heterojunction between two crystals, where a step $\Delta E_C$ or $\Delta E_V$ blocks one carrier by a factor $\exp(-\Delta E/kT)$ (double heterostructures of LEDs and lasers, window and back-surface-field layers, buffer layers of thin-film cells);
* **induced carriers** — a deposited film whose work function or fixed charge bends the absorber's bands into accumulation or inversion behind a thin passivating interlayer (poly-Si/SiO$_x$, a-Si:H, metal oxides, organic transport layers).

Most devices combine two. Two figures of merit describe a contact: the **contact resistivity** $\rho_c$ (mΩ cm$^2$) for the carrier it passes, and the **recombination parameter** $J_0$ (fA cm$^{-2}$) for the carrier it blocks; Brendel & Peibst (IEEE J. Photovolt. 6, 1413 (2016)) combine them into a selectivity $S_{10} = \log_{10}[(kT/q)/(\rho_c J_0)]$. They are measured on different structures — a symmetric passivated sample gives $J_0$ and the implied $V_{oc}$ but says nothing about $\rho_c$; a transfer-length (TLM) structure gives $\rho_c$; a finished cell gives $V_{oc}$, and the difference $\Delta V = iV_{oc} - V_{oc}$ is the voltage lost to imperfect selectivity (Chen, Mullapudi, Davis & Banerjee, Appl. Phys. Rev. 12, 041330 (2025), Tables I and III). A paper that reports one number without the other has characterized half a contact. For LEDs the equivalent quantities are the injection efficiency, the role of the electron-blocking layer, the $p$-contact resistivity, and the excess forward voltage $V_f - h\nu/q$ at the rated current.

### 3.1 Inventory of the junctions in your corpus (12 points)

Add the following columns to `survey.csv` for your top-20 rows (validated together with Task 2.4 above):

| column | content |
|---|---|
| `contact_geometry` | `top-bottom`, `top-top` (both on the illuminated/emitting side), `bottom-bottom` (both on the rear, e.g. IBC), `not reported`, `not applicable` |
| `e_contact_stack` | the electron-selective contact, layer by layer with thicknesses and process, e.g. `n+ poly-Si 100 nm (LPCVD, P-implanted) / SiOx 1.5 nm (thermal)` |
| `e_contact_route` | `doping`, `band offsets`, `induced carriers`, `doping + induced carriers`, `band offsets + doping`, `not reported` |
| `h_contact_stack` | the hole-selective contact, same format |
| `h_contact_route` | as above |
| `rho_c_mohmcm2` | contact resistivity if reported (state which contact), else `not reported` |
| `J0_fAcm2` | recombination parameter if reported (state which contact), else `not reported` |
| `other_contact_metric` | any other selectivity-related measurement: $iV_{oc}$, $\Delta V$, $S_{10}$, injection efficiency, $V_f$ at rated current, TLM or symmetric-sample used — or `not reported` |

**Write the narrative** (8 points, report): describe the electron and hole contacts your field uses — where they sit, what they are made of, how they are formed and at what thermal budget, and by which mechanism each is selective — and synthesize across the corpus: is there a dominant design, what are the alternatives, and what has changed over the years covered by your survey? State which papers report $\rho_c$, $J_0$ or their equivalents and which report neither, and what that means for comparing the designs.


**Task 3.1 Answer** *(4 points)*

**Dominant electron contact (stack, route) and hole contact (stack, route) in your corpus, with the bibtex keys of the papers that describe them best:** *(...)*

**Papers reporting both $\rho_c$ and $J_0$ (or the LED equivalents), and the values:** *(...)*


### 3.2 Band diagram of a representative device (18 points)

Construct the **equilibrium** band diagram of a representative version of your device: the absorber or emitter with the doping and thickness most common in your corpus, its electron-selective contact and its hole-selective contact, and the metals. The diagram is drawn by the procedure that Anderson introduced for heterojunctions (Solid-State Electron. 5, 341 (1962)) and that Rockett sets out step by step in *The Materials Science of Semiconductors* (Springer, 2008), §3.4.1 and Fig. 3.21:

1. **Materials apart.** Draw a common vacuum level. For each layer, place $E_C$ a distance $\chi$ (the electron affinity) below it and $E_V$ a further $E_g$ below; place that layer's Fermi level where its doping puts it. The band-edge positions relative to vacuum are properties of the materials and do not depend on doping; the work function $\Phi = E_{vac} - E_F$ does.
2. **Fix the offsets at the interfaces.** $\Delta E_C = \chi_1 - \chi_2$ and $\Delta E_V = \Delta E_g - \Delta E_C$ are pinned to the interface plane and survive whatever bending follows. Use measured offsets from your corpus where they exist; the electron-affinity rule is a first estimate only.
3. **Bring the layers into contact.** Electrons flow from the layer with the higher Fermi level to the lower until one Fermi level runs through the whole stack. The uncovered dopant ions (or the charge transferred to a film or metal) bend the bands, by exactly the difference in work functions in total, $qV_{bi} = \Phi_1 - \Phi_2$.
4. **Divide the bending between the sides.** Far from an interface each layer is unchanged. In the depletion approximation the potential drops divide as $V_1/V_2 = \varepsilon_2 N_2/\varepsilon_1 N_1$ (Rockett Eq. 3.42), so the bending falls on the lightly doped side, and the depletion widths follow from Poisson's equation.
5. **Read off the features.** Where an offset and the bending have the same sense a *spike* appears (crossed by thermionic emission or, if thin, by tunneling); where they oppose, the offset adds to one carrier's barrier and suppresses its current by $\exp(-\Delta E/kT)$. A metal contact brings the vacuum level to its work function at the interface (barrier height $\Phi_m - \chi$ for electrons, before Fermi-level pinning), and a heavily doped layer under it makes that barrier thin enough to tunnel.

The module below does steps 1–2 for you (`flatband_diagram`) and, optionally, steps 3–5 numerically (`equilibrium_diagram`, a Poisson solver with Fermi–Dirac statistics). **The graded deliverable is your own drawn diagram**, constructed by the five steps, in the report; the solver is there to check that your bending has the right sense and magnitude, and its output may be included as a second figure. The solver knows nothing about interface states, dipoles, gap states in amorphous layers, or bias and illumination; when it disagrees with the papers in your corpus, say why in your caption.

**Required content of the figure and caption:** the layer sequence with thicknesses (not to scale is fine, say so); $E_C$, $E_V$ and a single flat $E_F$; the vacuum level or the offsets $\Delta E_C$, $\Delta E_V$ labelled at each heterointerface; the parameters you used ($\chi$, $E_g$, doping type and level for each layer, metal work functions) with citations; arrows showing where electrons and holes exit; and the mechanism (doping / band offsets / induced carriers) named for each contact. For an LED, draw it in equilibrium as above and add one sentence on what changes under forward bias (the quasi-Fermi levels split; the electron-blocking layer's job). For a photodetector, state the bias polarity used in operation.


In [ ]:
# ============================================================
# HW02 MODULE C — one-dimensional band diagrams (instructor-provided; run, do not modify)
# 1) flatband_diagram(): the "apart" picture — vacuum levels aligned, bands flat, each layer's Fermi level
#    where its doping puts it (steps 1–2 of the construction in Rockett, Materials Science of Semiconductors, §3.4.1).
# 2) equilibrium_diagram(): solves Poisson's equation for the same stack in thermal equilibrium (one Fermi level),
#    Fermi–Dirac statistics, abrupt interfaces with the band offsets fixed by the electron affinities (Anderson,
#    Solid-State Electron. 5, 341 (1962)). Ends are either 'neutral' (bulk continues) or a metal work function.
# Equilibrium only: no bias, no illumination, no interface states or dipoles — see the assignment text.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

kT = 0.025852          # eV at 300 K
q_C = 1.602176634e-19  # C
eps0 = 8.8541878128e-14  # F/cm

# Reference parameters (300 K). Nc, Nv in cm^-3; chi = electron affinity (eV); eps = relative permittivity.
# Sources: Si — Green, J. Appl. Phys. 67, 2944 (1990); III–V — Vurgaftman et al. 2001; nitrides — Vurgaftman & Meyer 2003
# (chi of GaN ~4.1 eV from photoemission literature); CdTe/CdS — Madelung 2004; a-Si:H, poly-Si, oxides — typical
# device-simulation values. Treat these as starting points and replace them with values from your own sources.
MATERIALS = {
    'Si':      dict(Eg=1.12, chi=4.05, eps=11.7, Nc=2.86e19, Nv=3.10e19),
    'poly-Si': dict(Eg=1.12, chi=4.05, eps=11.7, Nc=2.86e19, Nv=3.10e19),
    'a-Si:H':  dict(Eg=1.72, chi=3.90, eps=11.9, Nc=1.0e20,  Nv=1.0e20),
    'SiO2':    dict(Eg=9.0,  chi=0.95, eps=3.9,  Nc=1.0e19,  Nv=1.0e19),
    'Ge':      dict(Eg=0.66, chi=4.00, eps=16.0, Nc=1.04e19, Nv=6.0e18),
    'GaAs':    dict(Eg=1.42, chi=4.07, eps=12.9, Nc=4.7e17,  Nv=9.0e18),
    'Al0.3Ga0.7As': dict(Eg=1.80, chi=3.83, eps=12.2, Nc=6.5e17, Nv=1.0e19),
    'AlAs':    dict(Eg=2.16, chi=3.50, eps=10.1, Nc=1.5e19,  Nv=1.7e19),
    'InP':     dict(Eg=1.34, chi=4.38, eps=12.5, Nc=5.7e17,  Nv=1.1e19),
    'In0.53Ga0.47As': dict(Eg=0.74, chi=4.50, eps=13.9, Nc=2.1e17, Nv=7.7e18),
    'Ga0.5In0.5P': dict(Eg=1.90, chi=4.10, eps=11.8, Nc=1.3e18, Nv=1.3e19),
    'GaN':     dict(Eg=3.42, chi=4.10, eps=8.9,  Nc=2.3e18,  Nv=4.6e19),
    'In0.15Ga0.85N': dict(Eg=2.80, chi=4.45, eps=9.5, Nc=2.0e18, Nv=4.0e19),
    'Al0.2Ga0.8N': dict(Eg=3.95, chi=3.75, eps=8.7, Nc=3.0e18, Nv=5.0e19),
    'CdTe':    dict(Eg=1.50, chi=4.28, eps=10.4, Nc=8.0e17,  Nv=1.8e19),
    'CdS':     dict(Eg=2.42, chi=4.50, eps=9.0,  Nc=2.2e18,  Nv=1.8e19),
    'MgZnO':   dict(Eg=3.60, chi=4.10, eps=9.0,  Nc=2.2e18,  Nv=1.8e19),
    'CIGS':    dict(Eg=1.15, chi=4.35, eps=13.6, Nc=2.2e18,  Nv=1.8e19),
    'MAPbI3':  dict(Eg=1.60, chi=3.90, eps=25.0, Nc=2.0e18,  Nv=2.0e18),
    'SnO2':    dict(Eg=3.60, chi=4.20, eps=9.0,  Nc=2.0e18,  Nv=2.0e18),
    'TiO2':    dict(Eg=3.20, chi=4.00, eps=30.0, Nc=1.0e19,  Nv=1.0e19),
    'MoOx':    dict(Eg=3.0,  chi=6.60, eps=12.0, Nc=1.0e19,  Nv=1.0e19),
    'NiOx':    dict(Eg=3.6,  chi=1.80, eps=11.0, Nc=1.0e19,  Nv=1.0e19),
    'C60':     dict(Eg=2.3,  chi=4.00, eps=4.0,  Nc=1.0e20,  Nv=1.0e20),
    'spiro-OMeTAD': dict(Eg=3.0, chi=2.10, eps=3.0, Nc=1.0e20, Nv=1.0e20),
}


def layer(material, thickness_nm, doping='i', N=0.0, name=None, **override):
    """Build a layer dict. doping: 'n', 'p' or 'i'; N = ionized dopant density (cm^-3)."""
    if isinstance(material, str):
        if material not in MATERIALS:
            raise KeyError(f"'{material}' is not in MATERIALS; pass a dict(Eg=, chi=, eps=, Nc=, Nv=) instead, or add it.")
        p = dict(MATERIALS[material]); base = material
    else:
        p = dict(material); base = name or 'custom'
    p.update(override)
    p.update(dict(t=float(thickness_nm), doping=doping, N=float(N), name=name or f"{base} ({doping}{'' if doping=='i' else f', {N:.0e}'})"))
    return p


# ---- Fermi–Dirac integral F_1/2 (Bednarczyk & Bednarczyk, Phys. Lett. A 64, 409 (1978); accurate to 0.4 %)
def fermi_half(eta):
    """Normalized Fermi-Dirac integral of order 1/2 (-> exp(eta) for eta << 0)."""
    eta = np.clip(np.asarray(eta, dtype=float), -700, 200)
    nu = eta ** 4 + 50.0 + 33.6 * eta * (1 - 0.68 * np.exp(-0.17 * (eta + 1) ** 2))
    return 1.0 / (np.exp(-eta) + 0.75 * np.sqrt(np.pi) * nu ** (-0.375))


def _neutral_level(p):
    """Position of E_F relative to the vacuum level (V = E_vac - E_F, eV) in a neutral bulk of this layer."""
    ND = p['N'] if p['doping'] == 'n' else 0.0; NA = p['N'] if p['doping'] == 'p' else 0.0
    def charge(V):   # V = E_vac - E_F ; Ec - E_F = V - chi ; E_F - Ev = chi + Eg - V
        n = p['Nc'] * fermi_half(-(V - p['chi']) / kT)
        pp = p['Nv'] * fermi_half(-(p['chi'] + p['Eg'] - V) / kT)
        return pp - n + ND - NA
    lo, hi = p['chi'] - 1.0, p['chi'] + p['Eg'] + 1.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if charge(mid) > 0: hi = mid      # charge rises monotonically with V (E_F moving down): too positive -> V too large
        else: lo = mid
    return 0.5 * (lo + hi)


def flatband_diagram(layers, left=None, right=None, ax=None, show_vacuum=True):
    """Draw the layers side by side with a common vacuum level (steps 1–2 of the construction)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.6 * len(layers) + 3, 4.5))
    x = 0.0; gap = 0.35
    for p in layers:
        w = 1.0
        Ec, Ev = -p['chi'], -p['chi'] - min(p['Eg'], 4.5)      # wide-gap insulators drawn truncated at 4.5 eV
        EF = -_neutral_level(p)
        if p['Eg'] > 4.5: ax.text(x + 0.5, Ev + 0.1, f"Eg = {p['Eg']} eV (truncated)", ha='center', fontsize=6.5, color='0.4')
        ax.plot([x, x + w], [0, 0], color='0.5', ls='--', lw=1)
        ax.plot([x, x + w], [Ec, Ec], 'k', lw=2); ax.plot([x, x + w], [Ev, Ev], 'k', lw=2)
        ax.fill_between([x, x + w], Ev - 0.6, Ev, color='0.85', lw=0)
        ax.plot([x, x + w], [EF, EF], color='C0', ls='-.', lw=1.4)
        ax.text(x + w / 2, Ec + 0.08, f"χ = {p['chi']:.2f}", ha='center', fontsize=7)
        ax.text(x + w / 2, Ev - 0.3, p['name'], ha='center', fontsize=7.5, va='top')
        ax.text(x + w + 0.02, EF, r'$E_F$', color='C0', fontsize=7, va='center')
        x += w + gap
    for side, wf, xpos in ((left, left, -0.6), (right, right, x - gap + 0.1)):
        if isinstance(wf, (int, float)):
            ax.plot([xpos, xpos + 0.5], [-wf, -wf], color='C0', ls='-.', lw=1.4)
            ax.fill_between([xpos, xpos + 0.5], -wf - 3, -wf, color='0.6', lw=0)
            ax.text(xpos + 0.25, -wf - 0.15, f'metal\nΦ = {wf} eV', ha='center', va='top', fontsize=7)
    ax.text(0, 0.08, r'$E_{vac}$ (aligned)', fontsize=8, color='0.4')
    ax.set_ylabel('Energy Relative to Vacuum (eV)'); ax.set_xticks([]); ax.grid(False)
    ax.set_title('Steps 1–2: Layers Apart, Vacuum Levels Aligned, Bands Flat', fontsize=10)
    return ax


def _mesh(layers, pts_per_layer=200, refine=8.0):
    """Nonuniform mesh, denser near interfaces. Returns x (cm) and layer index of each node."""
    xs, idx = [], []
    x0 = 0.0
    for i, p in enumerate(layers):
        t = p['t'] * 1e-7                                  # cm
        # points clustered toward both ends: s in (0,1) via a tanh stretch
        s = np.linspace(-1, 1, pts_per_layer)
        u = 0.5 * (1 + np.tanh(refine * s / 2) / np.tanh(refine / 2))
        xl = x0 + u * t
        if i > 0: xl = xl[1:]                              # share interface node
        xs.append(xl); idx.append(np.full(len(xl), i))
        x0 += t
    return np.concatenate(xs), np.concatenate(idx)


def solve_equilibrium(layers, left='neutral', right='neutral', pts_per_layer=200, verbose=True):
    """Nonlinear Poisson solver. left/right: 'neutral' or a metal work function (eV).
    Returns dict with x (nm), Evac, Ec, Ev (eV, E_F = 0), n, p (cm^-3), rho (C/cm^3), layer index."""
    x, li = _mesh(layers, pts_per_layer)
    N = len(x)
    chi = np.array([layers[i]['chi'] for i in li]); Eg = np.array([layers[i]['Eg'] for i in li])
    Nc = np.array([layers[i]['Nc'] for i in li]); Nv = np.array([layers[i]['Nv'] for i in li])
    eps = np.array([layers[i]['eps'] for i in li]) * eps0
    ND = np.array([layers[i]['N'] if layers[i]['doping'] == 'n' else 0.0 for i in li])
    NA = np.array([layers[i]['N'] if layers[i]['doping'] == 'p' else 0.0 for i in li])
    # unknown V = E_vac - E_F (eV) at each node. Poisson: d/dx(eps dV/dx) = q*(p - n + ND - NA)  [V in eV == volts]
    def carriers(V):
        n = Nc * fermi_half(-(V - chi) / kT); p = Nv * fermi_half(-(chi + Eg - V) / kT); return n, p
    def dcarriers(V):        # derivatives dn/dV, dp/dV (numerical, robust)
        h = 1e-4; n1, p1 = carriers(V + h); n0, p0 = carriers(V - h); return (n1 - n0) / (2 * h), (p1 - p0) / (2 * h)
    V = np.array([_neutral_level(layers[i]) for i in li])            # initial guess: each layer neutral
    # smooth the initial guess a little to help Newton
    for _ in range(3): V[1:-1] = 0.25 * V[:-2] + 0.5 * V[1:-1] + 0.25 * V[2:]
    VL = left if isinstance(left, (int, float)) else _neutral_level(layers[0])
    VR = right if isinstance(right, (int, float)) else _neutral_level(layers[-1])
    dx = np.diff(x)
    epsm = 0.5 * (eps[1:] + eps[:-1])                     # permittivity on half-points
    for it in range(200):
        n, p = carriers(V); dn, dp = dcarriers(V)
        rho = q_C * (p - n + ND - NA)
        F = np.zeros(N); J_main = np.zeros(N); J_lo = np.zeros(N - 1); J_up = np.zeros(N - 1)
        # interior: (eps_{i+1/2}(V_{i+1}-V_i)/dx_i - eps_{i-1/2}(V_i-V_{i-1})/dx_{i-1}) / ((dx_i+dx_{i-1})/2) - rho_i = 0
        hm = 0.5 * (dx[1:] + dx[:-1])
        a = epsm[:-1] / dx[:-1] / hm; c = epsm[1:] / dx[1:] / hm
        F[1:-1] = a * V[:-2] - (a + c) * V[1:-1] + c * V[2:] - rho[1:-1]
        J_main[1:-1] = -(a + c) - q_C * (dp[1:-1] - dn[1:-1]); J_lo[:-1] = a; J_up[1:] = c
        F[0] = V[0] - VL; J_main[0] = 1.0; F[-1] = V[-1] - VR; J_main[-1] = 1.0
        J_up[0] = 0.0; J_lo[-1] = 0.0
        Jm = diags([J_lo, J_main, J_up], [-1, 0, 1], format='csc')
        dV = spsolve(Jm, -F)
        step = np.clip(dV, -0.5, 0.5)                     # damping in eV
        V += step
        if np.max(np.abs(dV)) < 1e-6:
            break
    if verbose:
        print(f'Poisson solver converged in {it + 1} Newton iterations (max update {np.max(np.abs(dV)):.1e} eV).')
    n, p = carriers(V)
    return dict(x_nm=x * 1e7, Evac=V, Ec=V - chi, Ev=V - chi - Eg, n=n, p=p, rho=q_C * (p - n + ND - NA), layer=li,
                layers=layers, left=left, right=right)


def _draw(sol, layers, left, right, ax, show_vacuum, xlim=None, title=None):
    x = sol['x_nm']
    ax.plot(x, sol['Ec'], 'k', lw=2, label=r'$E_C$'); ax.plot(x, sol['Ev'], 'k', lw=2, label=r'$E_V$')
    ax.fill_between(x, sol['Ev'] - 20, sol['Ev'], color='0.88', lw=0)
    ax.axhline(0, color='C0', ls='-.', lw=1.4, label=r'$E_F$')
    if show_vacuum: ax.plot(x, sol['Evac'], color='0.5', ls='--', lw=1, label=r'$E_{vac}$')
    edges = np.cumsum([0] + [p['t'] for p in layers])
    semi = [i for i, p in enumerate(layers) if p['Eg'] < 4.5] or list(range(len(layers)))   # ignore insulators for the y-range
    m = np.isin(sol['layer'], semi)
    if xlim is not None: m &= (x >= xlim[0]) & (x <= xlim[1])
    ylo, yhi = sol['Ev'][m].min() - 0.7, sol['Ec'][m].max() + 0.9
    for i, p in enumerate(layers):
        ax.axvspan(edges[i], edges[i + 1], color=f'C{i % 10}', alpha=0.07, lw=0)
        span = (xlim[1] - xlim[0]) if xlim else edges[-1]
        if xlim is None or (edges[i + 1] > xlim[0] and edges[i] < xlim[1]):
            xc = 0.5 * (max(edges[i], xlim[0] if xlim else 0) + min(edges[i + 1], xlim[1] if xlim else edges[-1]))
            ax.text(xc, yhi - 0.05, p['name'], ha='center', va='top', fontsize=7.5,
                    rotation=0 if (edges[i + 1] - edges[i]) > 0.12 * span else 90)
    for wf, xpos, ha in ((left, edges[0], 'left'), (right, edges[-1], 'right')):
        if isinstance(wf, (int, float)):
            ax.axvline(xpos, color='0.3', lw=3); ax.text(xpos, yhi - 0.45, f' metal Φ = {wf} eV ', fontsize=7, ha=ha, va='top')
    ax.set_xlabel('Position (nm)'); ax.set_ylabel(r'Energy Relative to $E_F$ (eV)')
    ax.set_ylim(ylo, yhi)
    if xlim is not None: ax.set_xlim(*xlim)
    ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.25)
    if title: ax.set_title(title, fontsize=10)


def equilibrium_diagram(layers, left='neutral', right='neutral', pts_per_layer=200, show_vacuum=False, zoom=None, title=None):
    """Solve and plot the equilibrium band diagram (E_F = 0). zoom=(x1, x2) in nm adds a second panel with that
    range, which you will need whenever nanometer-thick contact layers sit on a micrometer-thick absorber.
    Returns the solution dict."""
    sol = solve_equilibrium(layers, left, right, pts_per_layer)
    if zoom is None:
        fig, ax = plt.subplots(figsize=(9, 4.6)); _draw(sol, layers, left, right, ax, show_vacuum, None, title or 'Equilibrium Band Diagram (Poisson Solution, One Fermi Level)')
    else:
        fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
        _draw(sol, layers, left, right, axs[0], show_vacuum, None, title or 'Equilibrium Band Diagram — Full Stack')
        _draw(sol, layers, left, right, axs[1], show_vacuum, zoom, 'Zoom: %g–%g nm' % zoom)
    plt.tight_layout()
    # report interface offsets and bending per layer
    print('Interfaces (band offsets fixed by electron affinities, Anderson 1962):')
    for i in range(len(layers) - 1):
        dEc = layers[i]['chi'] - layers[i + 1]['chi']; dEv = (layers[i + 1]['chi'] + layers[i + 1]['Eg']) - (layers[i]['chi'] + layers[i]['Eg'])
        print(f"  {layers[i]['name']} | {layers[i+1]['name']}:  ΔEc = {dEc:+.2f} eV, ΔEv = {dEv:+.2f} eV")
    print('Band bending inside each layer (max − min of E_C):')
    for i, p in enumerate(layers):
        m = sol['layer'] == i
        print(f"  {p['name']}: {sol['Ec'][m].max() - sol['Ec'][m].min():.3f} eV; E_C − E_F at its outer edge = {sol['Ec'][m][-1 if i == len(layers)-1 else 0]:+.3f} eV")
    return sol


def selectivity_report(sol, x_nm):
    """Print n, p and the conductivity-asymmetry ratio n/p at a position (nm) in the solved stack."""
    i = np.argmin(abs(sol['x_nm'] - x_nm)); n, p = sol['n'][i], sol['p'][i]
    print(f"x = {sol['x_nm'][i]:.1f} nm ({sol['layers'][sol['layer'][i]]['name']}): n = {n:.2e}, p = {p:.2e} cm^-3, n/p = {n/p:.1e}")

print('Module C loaded: layer, flatband_diagram, equilibrium_diagram, solve_equilibrium, selectivity_report. Materials:', ', '.join(MATERIALS))


In [ ]:
# YOUR CODE HERE (Task 3.2) — describe YOUR device as a list of layers, light-facing side first.
# layer(material, thickness_nm, doping='n'|'p'|'i', N=ionized dopant density cm^-3); materials are keys of MATERIALS
# (print(MATERIALS) to see them) or a dict(Eg=, chi=, eps=, Nc=, Nv=) with values from your sources.
# Thicknesses: use a few micrometres for a wafer absorber (not to scale) so the contact layers remain visible.
stack = [
    layer('poly-Si', 100, 'n', 1e20),     # example: n-type TOPCon cell, rear side last
    layer('SiO2', 1.5),
    layer('Si', 3000, 'n', 5e15),
    # ... boron-diffused front emitter would be a 'p' Si layer at the top; edit for your device
]
flatband_diagram(stack, left=4.3)                   # steps 1–2: vacuum levels aligned, bands flat


In [ ]:
# YOUR CODE HERE (Task 3.2) — steps 3–5 numerically (optional check of your hand-drawn diagram)
sol = equilibrium_diagram(stack, left=4.3, right='neutral', zoom=(0, 150))
selectivity_report(sol, 102)              # n, p and n/p at a position (nm): the conductivity asymmetry at that point
plt.savefig('band_diagram.png', dpi=200, bbox_inches='tight')


**Task 3.2 Answer** *(8 points)*

**Layer table used for the diagram** (material, thickness, $\chi$, $E_g$, doping, source for each value):

| layer | thickness | $\chi$ (eV) | $E_g$ (eV) | doping | source |
|---|---|---|---|---|---|
| ... | ... | ... | ... | ... | ... |

**Where the bending falls and why (step 4), and the spikes/steps you identified (step 5):** *(...)*

**Where the solver disagrees with your corpus (interface states, measured offsets, amorphous layers) and what you drew instead:** *(...)*

**Report status checklist:**

* ☐ Hand-constructed diagram with all required labels
* ☐ Caption with parameters and citations
* ☐ Mechanism named for each contact


### 3.3 Impact on performance (5 points, report)

Close the subsection with the properties → performance link: how the contacts limit the device in your corpus. For solar cells and photodetectors, use $\rho_c$ (series resistance, fill factor), $J_0$ or $\Delta V$ (open-circuit voltage, dark current) and, where the papers allow it, the parasitic absorption of the contact layers. For LEDs, use the forward voltage relative to $h\nu/q$, the injection efficiency and droop, and the $p$-contact. Where the numbers are not reported, say so and name the measurement that would have to be made.


---
## AI Use Disclosure

Per the course AI policy, disclose specifically where and how generative AI was used in completing this assignment (tool, purpose, and which tasks) — including Gemini in Colab for code and any tool used to search or summarize literature. If you did not use AI, state that.

*Your disclosure here (double-click to edit).*


## Submission Checklist

1. **Runtime → Restart and run all** — the notebook runs top to bottom with no errors (the survey validation cell must print `Validation PASSED`). Remember that the runtime is ephemeral: re-upload `survey.csv` (and any n,k CSV) after a restart.
2. All `# YOUR CODE HERE` cells are implemented for *your* material and device, and all *Your answer here* prompts are answered.
3. The AI Use Disclosure above is completed.
4. **File → Print → Save as PDF** — `EMA-6611-HW02-LastName.pdf`. Check that the figures are visible (the 3D viewer will not print; your screenshot is in the report).
5. **File → Download → Download .ipynb** — the notebook file itself.
6. In Overleaf: **Recompile**, check for `[?]` and overfull tables, then **Download PDF** — `LastName-Report-HW02.pdf`.
7. Submit **four files** (notebook PDF, .ipynb, extended survey.csv, report PDF) on Webcourses.
8. Keep your copy in Drive — Homework 3 (Passivation) adds columns to the same survey.csv and the next report subsection.

## References cited in this assignment

* Anderson, R. L. (1962). Experiments on Ge–GaAs heterojunctions. *Solid-State Electron.* 5, 341–351.
* Brendel, R., Peibst, R. (2016). Contact selectivity and efficiency in crystalline silicon photovoltaics. *IEEE J. Photovolt.* 6, 1413–1420.
* Chen, C.-H., Mullapudi, G. S. R., Davis, K. O., Banerjee, P. (2025). Atomic layer deposition (ALD) of passivating, carrier-selective oxides for silicon photovoltaics. *Appl. Phys. Rev.* 12, 041330.
* Jain, A. et al. (2013). The Materials Project: a materials genome approach to accelerating materials innovation. *APL Mater.* 1, 011002.
* Madelung, O. (2004). *Semiconductors: Data Handbook*, 3rd ed. Springer.
* Ong, S. P. et al. (2013). Python Materials Genomics (pymatgen): a robust, open-source Python library for materials analysis. *Comput. Mater. Sci.* 68, 314–319.
* Polyanskiy, M. N. (2024). Refractiveindex.info database of optical constants. *Sci. Data* 11, 94.
* Rego, N., Koes, D. (2015). 3Dmol.js: molecular visualization with WebGL. *Bioinformatics* 31, 2322–2324.
* Rockett, A. (2008). *The Materials Science of Semiconductors*. Springer, §3.4.
* Tauc, J., Grigorovici, R., Vancu, A. (1966). Optical properties and electronic structure of amorphous germanium. *Phys. Status Solidi* 15, 627–637.
* Vegard, L. (1921). Die Konstitution der Mischkristalle und die Raumfüllung der Atome. *Z. Phys.* 5, 17–26.
* Vogl, P., Hjalmarson, H. P., Dow, J. D. (1983). A semi-empirical tight-binding theory of the electronic structure of semiconductors. *J. Phys. Chem. Solids* 44, 365–378.
* Vurgaftman, I., Meyer, J. R., Ram-Mohan, L. R. (2001). Band parameters for III–V compound semiconductors and their alloys. *J. Appl. Phys.* 89, 5815–5875.
* Vurgaftman, I., Meyer, J. R. (2003). Band parameters for nitrogen-containing semiconductors. *J. Appl. Phys.* 94, 3675–3696.
* Würfel, U., Cuevas, A., Würfel, P. (2015). Charge carrier separation in solar cells. *IEEE J. Photovolt.* 5, 461–469.
* RDKit: Open-source cheminformatics, https://www.rdkit.org.
